# 📊 EDA — UltraEdit Region-Based 100K Dataset

Dataset: `BleachNick/UltraEdit_Region_Based_100k` | Split: `RegionBase` | Tổng: **108,179 mẫu**

### Cấu trúc Dataset
| Loại | Cột |
|------|------|
| 🖼️ Ảnh | `source_image`, `edited_image`, `mask_image` |
| 📝 Text | `source_caption`, `target_caption`, `edit_prompt`, `edit_object` |
| 📈 Metrics | `clip_sim_source`, `clip_sim_target`, `clip_sim_dir`, `clip_sim_image`, `dinov2_sim`, `ssim` |

In [ ]:
# ── Cell 1: Import thư viện ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datasets import load_dataset
from PIL import Image
from collections import Counter
import re
from IPython.display import display

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (12, 6), 'figure.dpi': 100})
print('✅ Thư viện đã sẵn sàng!')

In [ ]:
# ── Cell 2: Load Dataset ──────────────────────────────────────────────────────
ds = load_dataset('BleachNick/UltraEdit_Region_Based_100k')
dataset = ds['RegionBase']

print(f'✅ Tổng số mẫu: {len(dataset):,}')
print(f'📋 Các cột dữ liệu: {dataset.column_names}')
print()
# Lấy 1 mẫu thử để check
s = dataset[0]
print(f'📝 Edit object: {s["edit_object"]}')
print(f'📝 Edit prompt: {s["edit_prompt"]}')
print(f'📝 Source caption: {s["source_caption"]}')
print(f'📝 Target caption: {s["target_caption"]}')

In [ ]:
# ── Cell 3: Trực quan hóa mẫu dữ liệu ────────────────────────────────────────
# Xem 4 cặp (source | edited | mask) để hiểu dataset trông như thế nào
import random
random.seed(42)
indices = random.sample(range(len(dataset)), 4)

fig, axes = plt.subplots(4, 3, figsize=(16, 22))
fig.suptitle('Ví dụ mẫu dữ liệu UltraEdit (Source | Edited | Mask)', fontsize=16, fontweight='bold', y=1.01)

for row, idx in enumerate(indices):
    item = dataset[idx]
    axes[row, 0].imshow(item['source_image'])
    axes[row, 0].set_title(f'Source\n"{item["edit_object"]}"', fontsize=9)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(item['edited_image'])
    axes[row, 1].set_title(f'Edited\nPrompt: "{item["edit_prompt"][:50]}..."', fontsize=9)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(item['mask_image'], cmap='gray')
    axes[row, 2].set_title('Mask vùng sửa', fontsize=9)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: Xây dựng DataFrame (dùng sample nhỏ để tăng tốc độ) ──────────────
# Dùng 10,000 mẫu đầu để phân tích nhanh (bỏ limit nếu muốn toàn bộ)
SAMPLE_SIZE = 10_000
print(f'🔄 Đang trích xuất {SAMPLE_SIZE:,} mẫu để phân tích...')

records = []
for i, item in enumerate(dataset.select(range(SAMPLE_SIZE))):
    # Tính diện tích mask
    mask_arr = np.array(item['mask_image'])
    mask_pixels = np.count_nonzero(mask_arr > 128)
    total_pixels = mask_arr.shape[0] * mask_arr.shape[1]
    mask_ratio = mask_pixels / total_pixels
    
    # Tính chiều dài caption (số từ)
    src_cap = str(item['source_caption'])
    tgt_cap = str(item['target_caption'])
    edit_p  = str(item['edit_prompt'])
    
    records.append({
        'idx': item['idx'],
        'edit_object': item['edit_object'],
        'edit_prompt': edit_p,
        'source_caption': src_cap,
        'target_caption': tgt_cap,
        'src_word_count': len(re.findall(r'\b\w+\b', src_cap)),
        'tgt_word_count': len(re.findall(r'\b\w+\b', tgt_cap)),
        'edit_prompt_word_count': len(re.findall(r'\b\w+\b', edit_p)),
        'mask_ratio': mask_ratio,
        'clip_sim_source': item['clip_sim_source'],
        'clip_sim_target': item['clip_sim_target'],
        'clip_sim_dir': item['clip_sim_dir'],
        'clip_sim_image': item['clip_sim_image'],
        'dinov2_sim': item['dinov2_sim'],
        'ssim': item['ssim'],
    })

df = pd.DataFrame(records)
print(f'✅ DataFrame sẵn sàng: {df.shape[0]:,} hàng × {df.shape[1]} cột')
df.head(3)

In [ ]:
# ── Cell 5: Thống kê tổng quan ────────────────────────────────────────────────
print('📊 Thống kê mô tả các cột số:')
numeric_cols = ['src_word_count', 'tgt_word_count', 'edit_prompt_word_count',
                'mask_ratio', 'clip_sim_source', 'clip_sim_target',
                'clip_sim_dir', 'clip_sim_image', 'dinov2_sim', 'ssim']
df[numeric_cols].describe().round(4)

In [ ]:
# ── Cell 6: Phân tích Độ dài Caption ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Phân phối Độ dài Caption (số từ)', fontsize=14, fontweight='bold')

# Source Caption
sns.histplot(df['src_word_count'], bins=40, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Source Caption')
axes[0].set_xlabel('Số từ')
axes[0].axvline(df['src_word_count'].mean(), color='red', linestyle='--', label=f'Mean={df["src_word_count"].mean():.0f}')
axes[0].legend()

# Target Caption
sns.histplot(df['tgt_word_count'], bins=40, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Target Caption')
axes[1].set_xlabel('Số từ')
axes[1].axvline(df['tgt_word_count'].mean(), color='red', linestyle='--', label=f'Mean={df["tgt_word_count"].mean():.0f}')
axes[1].legend()

# Edit Prompt
sns.histplot(df['edit_prompt_word_count'], bins=30, kde=True, ax=axes[2], color='green')
axes[2].set_title('Edit Prompt')
axes[2].set_xlabel('Số từ')
axes[2].axvline(df['edit_prompt_word_count'].mean(), color='red', linestyle='--', label=f'Mean={df["edit_prompt_word_count"].mean():.0f}')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Phân tích Mask (diện tích vùng sửa) ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Phân tích Mask — Diện tích vùng cần sửa', fontsize=14, fontweight='bold')

# Histogram Mask Ratio
sns.histplot(df['mask_ratio'] * 100, bins=50, kde=True, ax=axes[0], color='purple')
axes[0].set_title('Phân phối tỉ lệ diện tích Mask')
axes[0].set_xlabel('Tỉ lệ (%) so với toàn ảnh')
axes[0].axvline(df['mask_ratio'].mean() * 100, color='red', linestyle='--',
                label=f'Mean={df["mask_ratio"].mean()*100:.1f}%')
axes[0].legend()

# Phân loại theo nhóm diện tích
bins = [0, 0.05, 0.15, 0.30, 0.50, 1.01]
labels = ['Rất nhỏ\n(<5%)', 'Nhỏ\n(5-15%)', 'Trung bình\n(15-30%)', 'Lớn\n(30-50%)', 'Rất lớn\n(>50%)']
df['mask_category'] = pd.cut(df['mask_ratio'], bins=bins, labels=labels)
mask_counts = df['mask_category'].value_counts().sort_index()
colors = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']
axes[1].pie(mask_counts, labels=mask_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Phân loại theo kích thước vùng sửa')

plt.tight_layout()
plt.show()
print(mask_counts)

In [ ]:
# ── Cell 8: Spatial HeatMap — Vùng sửa thường nằm ở đâu? ────────────────────
print('🔄 Đang tạo Spatial HeatMap (dùng 2,000 mask)...')
TARGET_SIZE = (256, 256)
heatmap = np.zeros(TARGET_SIZE, dtype=np.float64)

for item in dataset.select(range(2000)):
    mask = np.array(item['mask_image'].resize(TARGET_SIZE, Image.NEAREST))
    if mask.max() > 1:
        mask = (mask > 128).astype(np.float64)
    heatmap += mask

heatmap = heatmap / heatmap.max()  # Normalize 0-1

plt.figure(figsize=(8, 8))
plt.imshow(heatmap, cmap='hot', interpolation='bilinear')
plt.colorbar(label='Tần suất xuất hiện (normalized)')
plt.title('Spatial HeatMap — Vùng nào thường bị sửa nhất?\n(Tổng hợp từ 2,000 mask)', fontsize=13, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()
print('✅ Màu vàng/đỏ = vùng xuất hiện nhiều lần trong các mask')

In [ ]:
# ── Cell 9: Phân tích các chỉ số chất lượng (Quality Metrics) ────────────────
metrics = ['clip_sim_source', 'clip_sim_target', 'clip_sim_dir', 'clip_sim_image', 'dinov2_sim', 'ssim']
metric_labels = ['CLIP-Sim Source', 'CLIP-Sim Target', 'CLIP-Dir', 'CLIP-Img', 'DINOv2-Sim', 'SSIM']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Phân phối các chỉ số chất lượng chỉnh sửa', fontsize=14, fontweight='bold')
axes = axes.flatten()

colors = sns.color_palette('husl', len(metrics))
for i, (col, label) in enumerate(zip(metrics, metric_labels)):
    sns.histplot(df[col], bins=40, kde=True, ax=axes[i], color=colors[i])
    mean_val = df[col].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean={mean_val:.3f}')
    axes[i].set_title(label, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Score')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10: Correlation heatmap giữa các Metrics ────────────────────────────
corr_cols = ['mask_ratio', 'src_word_count', 'tgt_word_count', 'edit_prompt_word_count'] + metrics
corr = df[corr_cols].corr()

plt.figure(figsize=(12, 9))
mask_tri = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask_tri, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5, square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Ma trận tương quan giữa các đặc trưng', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 11: Phân tích Top Edit Object (Vật thể được sửa nhiều nhất) ─────────
# Làm sạch tên edit_object
def clean_object(obj):
    if not isinstance(obj, str):
        return 'Unknown'
    obj = obj.strip().lower()
    obj = re.sub(r'"|\'|,$', '', obj)
    return obj.title()

df['edit_object_clean'] = df['edit_object'].apply(clean_object)
top_objects = df['edit_object_clean'].value_counts().head(30)

plt.figure(figsize=(14, 8))
bars = sns.barplot(
    x=top_objects.values,
    y=top_objects.index,
    palette=sns.color_palette('YlOrRd_r', len(top_objects))
)
for i, (val, name) in enumerate(zip(top_objects.values, top_objects.index)):
    plt.text(val + 0.3, i, f'{val:,}', va='center', fontsize=8)

plt.title('Top 30 Vật thể được chỉnh sửa nhiều nhất (edit_object)', fontsize=14, fontweight='bold')
plt.xlabel('Số lần xuất hiện (trong 10K mẫu)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 12: Phân tích từ trong Edit Prompt (Action Word Cloud) ───────────────
# Tìm những động từ hành động xuất hiện nhiều trong edit_prompt
STOPWORDS = {'a','an','the','and','or','but','in','on','at','to','for','of','with',
             'by','from','into','this','that','it','is','are','was','were','be','been',
             'have','has','do','does','did','will','would','could','should','may','might',
             'its','their','there','they','he','she','we','who','which','what','when',
             'where','how','all','as','if','up','out','so','some','each','than','then'}

all_words = []
for prompt in df['edit_prompt']:
    words = re.findall(r'\b[a-zA-Z]+\b', str(prompt).lower())
    all_words.extend([w for w in words if w not in STOPWORDS and len(w) > 2])

word_freq = Counter(all_words)
top_words = word_freq.most_common(30)

words, counts = zip(*top_words)
palette = sns.color_palette('Blues_d', len(words))

plt.figure(figsize=(14, 6))
bars = plt.bar(words, counts, color=palette[::-1])
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.title('Top 30 từ xuất hiện nhiều nhất trong Edit Prompt', fontsize=14, fontweight='bold')
plt.ylabel('Số lần xuất hiện')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
             f'{count:,}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 13: Phân loại Task theo Edit Prompt (Keyword-based) ─────────────────
def classify_edit(prompt):
    p = str(prompt).lower()
    if re.search(r'\breplace\b|\bchange\b|\bswap\b|\bsubstitute\b', p):
        return 'Replace/Swap'
    elif re.search(r'\badd\b|\binsert\b|\bput\b|\bplace\b|\binclude\b', p):
        return 'Add Object'
    elif re.search(r'\bremove\b|\bdelete\b|\berase\b|\beliminate\b|\bget rid\b', p):
        return 'Remove Object'
    elif re.search(r'\bcolor\b|\bcolour\b|\bred\b|\bblue\b|\bgreen\b|\bblack\b|\bwhite\b|\byellow\b|\bpink\b|\bpurple\b|\borange\b', p):
        return 'Color Change'
    elif re.search(r'\btexture\b|\bstyle\b|\bpattern\b|\bmaterial\b|\bwood\b|\bmetal\b|\bstone\b', p):
        return 'Texture/Style'
    elif re.search(r'\bbackground\b|\bsky\b|\bscene\b|\benvironment\b', p):
        return 'Background'
    else:
        return 'Other'

df['edit_type'] = df['edit_prompt'].apply(classify_edit)
type_counts = df['edit_type'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Phân loại loại Task chỉnh sửa (Keyword-based)', fontsize=14, fontweight='bold')

# Pie chart
colors = sns.color_palette('Set2', len(type_counts))
ax1.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Tỉ lệ loại Task')

# Bar chart
sns.barplot(x=type_counts.values, y=type_counts.index, palette='Set2', ax=ax2)
for i, v in enumerate(type_counts.values):
    ax2.text(v + 20, i, f'{v:,}', va='center')
ax2.set_xlabel('Số lượng mẫu')
ax2.set_title('Phân phối loại Task')

plt.tight_layout()
plt.show()
print(type_counts)

In [ ]:
# ── Cell 14: Scatter Plot — Mask Area vs Metrics ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Mối quan hệ giữa kích thước Mask và chỉ số Metrics', fontsize=13, fontweight='bold')

scatter_cols = [('clip_sim_dir', 'CLIP-Dir', 'royalblue'),
                ('dinov2_sim', 'DINOv2-Sim', 'darkorange'),
                ('ssim', 'SSIM', 'green')]

sample_plot = df.sample(2000, random_state=42)
for i, (col, label, color) in enumerate(scatter_cols):
    axes[i].scatter(sample_plot['mask_ratio'] * 100, sample_plot[col],
                    alpha=0.3, s=8, color=color)
    # Thêm đường trend
    z = np.polyfit(sample_plot['mask_ratio'] * 100, sample_plot[col], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 100, 100)
    axes[i].plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend')
    
    corr_val = sample_plot['mask_ratio'].corr(sample_plot[col])
    axes[i].set_xlabel('Mask Area (%)')
    axes[i].set_ylabel(label)
    axes[i].set_title(f'Mask Ratio vs {label}\n(Pearson r = {corr_val:.3f})')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 15: Best vs Worst Samples (theo CLIP-Dir) ────────────────────────────
# CLIP-Dir đo mức độ chỉnh sửa có đúng hướng với prompt không
# Cao = Editing thành công đúng ý | Thấp = Editing thất bại hoặc sai hướng

top_k = 3
best_samples = df.nlargest(top_k, 'clip_sim_dir').index.tolist()
worst_samples = df.nsmallest(top_k, 'clip_sim_dir').index.tolist()

def show_samples(indices_list, title):
    fig, axes = plt.subplots(len(indices_list), 3, figsize=(15, 5*len(indices_list)))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    for row, df_idx in enumerate(indices_list):
        ds_idx = int(df.loc[df_idx, 'idx'])
        item = dataset[ds_idx]
        row_info = df.loc[df_idx]
        
        axes[row, 0].imshow(item['source_image'])
        axes[row, 0].set_title('Source', fontsize=9)
        axes[row, 0].axis('off')
        
        axes[row, 1].imshow(item['edited_image'])
        axes[row, 1].set_title(f'Edited ({row_info["edit_prompt"][:40]}...)', fontsize=8)
        axes[row, 1].axis('off')
        
        axes[row, 2].imshow(item['mask_image'], cmap='gray')
        axes[row, 2].set_title(
            f'Mask | CLIP-Dir={row_info["clip_sim_dir"]:.3f}\nSSIM={row_info["ssim"]:.3f}', fontsize=9)
        axes[row, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

show_samples(best_samples, '✅ Top 3 chỉnh sửa TỐT NHẤT (theo CLIP-Dir cao)')
show_samples(worst_samples, '❌ Top 3 chỉnh sửa KÉM NHẤT (theo CLIP-Dir thấp)')

In [ ]:
# ── Cell 16: Tóm tắt EDA ─────────────────────────────────────────────────────
print('=' * 60)
print('📊 TÓM TẮT EDA — UltraEdit Region-Based 100K')
print('=' * 60)
print(f'\n📦 Tổng số mẫu phân tích: {len(df):,}')
print(f'\n📝 TEXT ANALYSIS:')
print(f'  • Source Caption: {df["src_word_count"].mean():.0f} từ (avg), {df["src_word_count"].max()} từ (max)')
print(f'  • Target Caption: {df["tgt_word_count"].mean():.0f} từ (avg), {df["tgt_word_count"].max()} từ (max)')
print(f'  • Edit Prompt:    {df["edit_prompt_word_count"].mean():.0f} từ (avg), {df["edit_prompt_word_count"].max()} từ (max)')
print(f'\n🖼️  MASK ANALYSIS:')
print(f'  • Diện tích Mask trung bình: {df["mask_ratio"].mean()*100:.1f}% của ảnh')
print(f'  • Phần lớn là chỉnh sửa nhỏ (<15%): {(df["mask_ratio"] < 0.15).mean()*100:.1f}%')
print(f'\n📈 QUALITY METRICS:')
for col, label in zip(metrics, metric_labels):
    print(f'  • {label}: {df[col].mean():.4f} ± {df[col].std():.4f}')
print()
print('🏆 Chỉnh sửa tốt nhất (CLIP-Dir cao nhất):')
best = df.nlargest(1, 'clip_sim_dir').iloc[0]
print(f'  Edit: "{best["edit_prompt"]}" | CLIP-Dir={best["clip_sim_dir"]:.3f}')
print('=' * 60)